In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

# Generate KG

In [3]:
!python construct_kg.py

/home/damon/kacd_submission/construct_kg.py:17: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)
../clbp_causal_md/hu2022/hu2022.md
No sentence-transformers model found with name mistralai/Mistral-7B-Instruct-v0.3. Creating a new one with mean pooling.
Loading checkpoint shards: 100%|██████████████████| 3/3 [00:05<00:00,  1.97s/it]
Traceback (most recent call last):
  File "/home/damon/kacd_submission/construct_kg.py", line 54, in <module>
    chunks.extend(markdown_parser.semantic_chunk(content) )
  File "/home/damon/kacd_submission/util/markdown_parser.py", line 75, in semantic_chunk
    splitter =

# Generate Predictions

In [4]:
!python query_kg.py

/home/damon/kacd_submission/query_kg.py:40: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(
/shared/graphrag/lib/python3.10/site-packages/langchain_community/graphs/neo4j_graph.py:404: PreviewWarning: notifications_disabled_classifications is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(
/home/damon/kacd_submission/query_kg.py:93: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-hug

In [ ]:
!python query_llm.py

parallel_apply:   1%|▎                          | 2/145 [00:07<07:17,  3.06s/it]

In [ ]:
!python query_rag.py

In [ ]:
!python query_causal_lit_kg.py

In [ ]:
!python query_causal_lit_llm.py

In [ ]:
!python query_causal_lit_rag.py

# Combine all results

In [ ]:
kgrag = pd.read_csv("results/kgrag.csv", index_col=0).drop(columns=["Label"])
llm = pd.read_csv("results/llm.csv", index_col=0).drop(columns=["Label"])
rag = pd.read_csv("results/rag.csv", index_col=0).drop(columns=["Label"])

kgrag_cl = pd.read_csv("results/kg+rag_full_causal_literature.csv", index_col=0).drop(columns=["Label"])
llm_cl = pd.read_csv("results/llm_full_causal_literature.csv", index_col=0).drop(columns=["Label"])
llm_cl["Causal Literature Report"] = ''
rag_cl = pd.read_csv("results/llm+rag_full_causal_literature.csv", index_col=0).drop(columns=["Label"])

In [ ]:
from util.helpers import consolidate_causal_literature

# Apply to all causal literature dataframes
kgrag_cl = consolidate_causal_literature(kgrag_cl)
llm_cl = consolidate_causal_literature(llm_cl)
rag_cl = consolidate_causal_literature(rag_cl)

In [ ]:
kgrag = pd.merge(kgrag, kgrag_cl, on=["Var1", "Var2"])
rag = pd.merge(rag, rag_cl, on=["Var1", "Var2"])
llm = pd.merge(llm, llm_cl, on=["Var1", "Var2"])

In [ ]:
# Add context column to each dataframe
kgrag['context'] = 'kg+llm_full.csv'
llm['context'] = 'llm_full.csv'
rag['context'] = 'rag_full.csv'

# Concatenate all dataframes
consolidated = pd.concat([kgrag, llm, rag], ignore_index=True)

consolidated.to_csv("results/results_undirected_combined.csv")

# Generate directed results

In [ ]:
!python query_directions_without_proto.py

In [ ]:
!python query_directions_with_proto.py